# Path B: Step-by-Step Strand Exploration
## FC Mold G5 — TATA IJmuiden CC23

This notebook walks through the analysis pipeline **one step at a time**.  
Use it when you want to understand, debug, or experiment with individual steps.

**Imports from `src/`** — same code as Path A (`run_pipeline`). Fix once, works everywhere.

| Section | What it does |
| --- | --- |
| 1. Setup | Install deps, import from `src/`, select strand |
| 2. Data Loading | Load parquet, join boExpert+dtExpert, convert units |
| 3. Metadata Join | Attach quality labels, inspect coverage |
| 4. FC Mold Filtering | Filter to FC Mold active periods |
| 5. Sequence Identification | Sliding window → stable sequences |
| 6. Disturbance Detection | Classify mold level events |
| 7. Results & Visualization | Summary statistics, distributions, correlations |

In [0]:
%pip install mpl-scatter-density astropy --quiet

In [0]:
import sys, os

# Add project root to Python path (uses CWD — works in both Users and Repos)
project_root = os.getcwd()
assert os.path.isdir(os.path.join(project_root, "src")), (
    f"src/ not found in {project_root} — run from the project root"
)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Core pipeline modules
from src.config import AnalysisConfig, StrandConfig, CONFIG, STRAND_CONFIGS, METADATA_PATH
from src.data_loading import (
    add_plain_timestamp, get_expert_files, load_expert_files,
    convert_units, StrandDataLoader,
)
from src.sequence_analysis import (
    custom_rounding, segment_by_time_gaps, identify_sequences,
    identify_sequences_segmented, SequenceAnalyzer,
)
from src.disturbance_detection import (
    detect_excursion_event_robust, detect_slow_drift_event,
    detect_transient_bump_dynamic, detect_high_variability_event,
)

# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from pyspark.sql import functions as F

print(f"CONFIG: {CONFIG}")
print(f"Strands available: {list(STRAND_CONFIGS.keys())}")

In [0]:
# ============================================================
# CHANGE THIS to explore a different strand
# ============================================================
STRAND_ID = "23_6"  # Options: "23_6" or "23_5"
# ============================================================

strand_config = STRAND_CONFIGS[STRAND_ID]
print(f"Exploring: {strand_config.strand_name}")
print(f"Data path: {strand_config.data_path}")
print(f"EMBR columns: {strand_config.embr_current_cols}")

## 2. Data Loading
Load boExpert (2Hz FBG) and dtExpert (1Hz EMBR) from DBFS, aggregate, join, convert units.

In [0]:
# Discover boExpert and dtExpert files on DBFS
bo_files, dt_files = get_expert_files(strand_config.data_path)

print(f"boExpert files: {len(bo_files)}")
for f in bo_files[:3]:
    print(f"  {f}")
if len(bo_files) > 3:
    print(f"  ... and {len(bo_files)-3} more")

print(f"\ndtExpert files: {len(dt_files)}")
for f in dt_files[:3]:
    print(f"  {f}")
if len(dt_files) > 3:
    print(f"  ... and {len(dt_files)-3} more")

In [0]:
# Load raw Spark DataFrames
df_bo = load_expert_files(bo_files, spark)
df_dt = load_expert_files(dt_files, spark)

print(f"boExpert: {df_bo.count():,} rows, {len(df_bo.columns)} columns")
print(f"dtExpert: {df_dt.count():,} rows, {len(df_dt.columns)} columns")

print("\nboExpert columns:")
for c in sorted(df_bo.columns):
    print(f"  {c}")

In [0]:
# Aggregate boExpert by timestamp (mean for numerics, first for categoricals)
from pyspark.sql.types import NumericType

key_col = "plainTimeStamp"
numeric_cols = [f.name for f in df_bo.schema.fields
                if isinstance(f.dataType, NumericType) and f.name != key_col]
non_numeric_cols = [f.name for f in df_bo.schema.fields
                    if not isinstance(f.dataType, NumericType) and f.name != key_col]

agg_exprs = [F.avg(c).alias(c) for c in numeric_cols]
agg_exprs += [F.first(c).alias(c) for c in non_numeric_cols]

df_bo_agg = df_bo.groupBy(key_col).agg(*agg_exprs)
print(f"boExpert aggregated: {df_bo_agg.count():,} rows (was {df_bo.count():,})")

In [0]:
# Inner join on plainTimeStamp
df_joined = df_bo_agg.alias("bo").join(
    df_dt.alias("dt"), on="plainTimeStamp", how="inner"
).cache()

print(f"Joined DataFrame: {df_joined.count():,} rows")
print(f"Columns: {len(df_joined.columns)}")
print(f"Time range: {df_joined.select(F.min('plainTimeStamp'), F.max('plainTimeStamp')).first()}")

In [0]:
# Convert: m/s->m/min, m->mm, m3/s->L/min
df_converted = convert_units(df_joined)

# Quick sanity check
sample = df_converted.select("castingSpeed", "Mold Level", "SENImmersionDepth").summary("min", "mean", "max")
display(sample)

## 3. Metadata Join
Attach quality casting labels from metadata CSV. Range-join on timestamp boundaries.

In [0]:
# Load metadata (semicolon-delimited)
df_meta = spark.read.csv(METADATA_PATH, header=True, inferSchema=True, sep=";")

# Filter to this strand
strand_num = int(STRAND_ID.split("_")[1])
df_meta_strand = (
    df_meta
    .withColumn("ts_start", F.col("Datetime start first heat").cast("timestamp"))
    .withColumn("ts_end", F.col("Datetime start last heat").cast("timestamp"))
    .filter(F.col("Strand ID") == strand_num)
)

print(f"Metadata: {df_meta_strand.count()} casting periods for Strand {strand_num}")
display(df_meta_strand.select("ts_start", "ts_end", "Quality casting").limit(5))

In [0]:
# Range join: attach quality label to each data row
join_cond = (
    (df_converted["plainTimeStamp"] >= df_meta_strand["ts_start"]) &
    (df_converted["plainTimeStamp"] <= df_meta_strand["ts_end"])
)

df_with_meta = df_converted.join(
    df_meta_strand.select("ts_start", "ts_end", "Quality casting"),
    on=join_cond, how="left"
).drop("ts_start", "ts_end")

total = df_with_meta.count()
matched = df_with_meta.filter(F.col("Quality casting").isNotNull()).count()
print(f"Metadata coverage: {matched:,}/{total:,} rows ({100*matched/total:.1f}%)")

In [0]:
# Select analysis columns and convert to Pandas
cols = [
    "plainTimeStamp", "castingSpeed", "moldWidth", "SENImmersionDepth",
    "Mold Level", "Mold Level Sensor Left", "Mold Level Sensor Right",
    "Argon Flow SEN", "Argon Flow Stopper", "Quality casting",
] + strand_config.embr_current_cols

cols_available = [c for c in cols if c in df_with_meta.columns]

df = (
    df_with_meta.select(*cols_available)
    .dropna(subset=["castingSpeed", "moldWidth", "SENImmersionDepth"])
    .toPandas()
    .sort_values("plainTimeStamp")
    .reset_index(drop=True)
)

print(f"Pandas DataFrame: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Time: {df['plainTimeStamp'].min()} to {df['plainTimeStamp'].max()}")
print(f"\nColumn dtypes:")
print(df.dtypes.to_string())

## 4. FC Mold Active Filtering
Keep only rows where ALL EMBR currents are non-zero (electromagnetic braking is ON).

In [0]:
# FC Mold active = all EMBR currents non-zero
embr_cols = strand_config.embr_current_cols
mask_fc_on = (df[embr_cols] != 0).all(axis=1)

df_fc = df[mask_fc_on].reset_index(drop=True)

print(f"FC Mold active: {len(df_fc):,} / {len(df):,} rows ({100*len(df_fc)/len(df):.1f}%)")
print(f"\nCasting speed range: {df_fc['castingSpeed'].min():.2f} - {df_fc['castingSpeed'].max():.2f} m/min")
print(f"Mold width range: {df_fc['moldWidth'].min():.1f} - {df_fc['moldWidth'].max():.1f} mm")
print(f"SEN depth range: {df_fc['SENImmersionDepth'].min()*1000:.0f} - {df_fc['SENImmersionDepth'].max()*1000:.0f} mm")

## 5. Sequence Identification
Sliding window (300 samples = 6 min) to find stable casting periods.  
A sequence is NORMAL if casting speed varies < 0.1 m/min AND EMBR currents have no jumps > 50 A.

In [0]:
# Run the segmented sequence identification
import time

t0 = time.time()
normal_groups, abnormal_groups = identify_sequences_segmented(
    df_fc,
    Vc_column="castingSpeed",
    window_size=CONFIG.window_size,
    Vc_threshold=CONFIG.vc_threshold,
    Curr_columns=embr_cols,
    Curr_threshold=CONFIG.curr_threshold,
    tcol="plainTimeStamp",
    max_gap_seconds=CONFIG.max_gap_seconds,
    min_segment_len=CONFIG.window_size,
)
elapsed = time.time() - t0

print(f"Sequence identification completed in {elapsed:.1f}s")
print(f"  Normal sequences: {len(normal_groups)}")
print(f"  Abnormal sequences: {len(abnormal_groups)}")
print(f"  Samples per normal sequence: {CONFIG.window_size} (= {CONFIG.window_size}s at 1Hz)")

In [0]:
# Compute statistics for each normal sequence
records = []
for i, idx_list in enumerate(normal_groups):
    seg = df_fc.iloc[idx_list]
    rec = {
        "seq_id": i,
        "start_time": seg["plainTimeStamp"].iloc[0],
        "end_time": seg["plainTimeStamp"].iloc[-1],
        "n_samples": len(seg),
        "castingSpeed_mean": seg["castingSpeed"].mean(),
        "moldWidth_mean": seg["moldWidth"].mean(),
        "SEN_depth_mean": seg["SENImmersionDepth"].mean(),
        "ML_std": seg["Mold Level"].std(),
        "ML_mean": seg["Mold Level"].mean(),
        "ptp_mm": seg["Mold Level"].max() - seg["Mold Level"].min(),
        "Quality_casting": seg["Quality casting"].mode().iloc[0] if not seg["Quality casting"].mode().empty else np.nan,
    }
    # Add EMBR current means
    for col in embr_cols:
        rec[f"{col}_mean"] = seg[col].mean()
    records.append(rec)

df_seq = pd.DataFrame(records)

print(f"Per-sequence statistics: {len(df_seq)} sequences")
print(f"\nMold Level sigma distribution:")
print(df_seq["ML_std"].describe().to_string())
print(f"\nStable (sigma < {CONFIG.ml_stability_threshold_mm} mm): "
      f"{(df_seq['ML_std'] < CONFIG.ml_stability_threshold_mm).sum()} / {len(df_seq)} "
      f"({100*(df_seq['ML_std'] < CONFIG.ml_stability_threshold_mm).mean():.1f}%)")

## 6. Disturbance Detection
Classify each sequence's mold level signal:  
**Excursion** (>8mm, >5s) | **Slow Drift** (monotonic >60s, >10mm) | **Transient Bump** (spike >8x noise) | **High Variability** (ptp>10mm or >10% outside band) | **Normal**

In [0]:
# Run all 4 detectors on each sequence
# src/ detectors take a 1-D numpy signal array and return bool
disturbance_labels = []

for i, idx_list in enumerate(normal_groups):
    seg = df_fc.iloc[idx_list]
    signal = seg["Mold Level"].values
    flags = []
    
    if detect_excursion_event_robust(
        signal,
        threshold_mm=CONFIG.excursion_threshold_mm,
        min_duration_s=CONFIG.excursion_min_duration_s,
    ):
        flags.append("Excursion")
    
    if detect_slow_drift_event(
        signal,
        min_run_s=CONFIG.slow_drift_min_run_s,
        min_amplitude_mm=CONFIG.slow_drift_min_amplitude_mm,
    ):
        flags.append("Slow drift")
    
    if detect_transient_bump_dynamic(
        signal,
        k_amp=CONFIG.transient_bump_k_amp,
        min_amp_mm=CONFIG.transient_bump_min_mm,
    ):
        flags.append("Transient bump")
    
    if detect_high_variability_event(
        signal,
        ptp_threshold_mm=CONFIG.high_var_ptp_threshold_mm,
        band_mm=CONFIG.high_var_band_mm,
        fraction_threshold=CONFIG.high_var_fraction_threshold,
    ):
        flags.append("High variability")
    
    label = " + ".join(flags) if flags else "Normal"
    disturbance_labels.append(label)

df_seq["disturbance_type"] = disturbance_labels

# Summary
print("Disturbance classification:")
print(df_seq["disturbance_type"].value_counts().to_string())
print(f"\nClean (Normal) sequences: {(df_seq['disturbance_type'] == 'Normal').sum()} / {len(df_seq)}")

## 7. Results and Visualization
Summary plots: ML sigma distribution, parameter correlations, disturbance breakdown.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of ML sigma
ax = axes[0]
ax.hist(df_seq["ML_std"], bins=50, color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(CONFIG.ml_stability_threshold_mm, color="red", ls="--", lw=2,
           label=f"Threshold = {CONFIG.ml_stability_threshold_mm} mm")
ax.set_xlabel("Mold Level sigma (mm)")
ax.set_ylabel("Count")
ax.set_title(f"{strand_config.strand_name}: ML Sigma Distribution")
ax.legend()

# Box plot by disturbance type
ax = axes[1]
types = df_seq["disturbance_type"].value_counts().index[:6]  # Top 6
data_for_box = [df_seq[df_seq["disturbance_type"]==t]["ML_std"].values for t in types]
ax.boxplot(data_for_box, labels=[t[:15] for t in types], vert=True)
ax.set_ylabel("ML sigma (mm)")
ax.set_title("ML Sigma by Disturbance Type")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
display(fig)
plt.close(fig)

In [0]:
# Clean sequences only
df_clean = df_seq[df_seq["disturbance_type"] == "Normal"].copy()
print(f"Clean sequences: {len(df_clean)}")

# Scatter: ML sigma vs key parameters
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

params = [
    ("castingSpeed_mean", "Casting Speed (m/min)"),
    ("moldWidth_mean", "Mold Width (mm)"),
    ("SEN_depth_mean", "SEN Depth (m)"),
    ("ptp_mm", "Peak-to-Peak (mm)"),
]

for ax, (col, label) in zip(axes.flat, params):
    ax.scatter(df_clean[col], df_clean["ML_std"], alpha=0.4, s=10, c="steelblue")
    ax.set_xlabel(label)
    ax.set_ylabel("ML sigma (mm)")
    ax.axhline(CONFIG.ml_stability_threshold_mm, color="red", ls="--", alpha=0.5)
    # Simple R-squared
    from scipy.stats import pearsonr
    r, _ = pearsonr(df_clean[col].dropna(), df_clean["ML_std"].dropna())
    ax.set_title(f"{label} vs ML sigma (R={r:.3f})")

plt.suptitle(f"{strand_config.strand_name} — Parameter Correlations (Clean)", fontsize=13)
plt.tight_layout()
display(fig)
plt.close(fig)

In [0]:
# Final summary
print("=" * 60)
print(f"EXPLORATION SUMMARY: {strand_config.strand_name}")
print("=" * 60)
print(f"")
print(f"Raw data rows:        {len(df):,}")
print(f"FC Mold active rows:  {len(df_fc):,} ({100*len(df_fc)/len(df):.1f}%)")
print(f"Stable sequences:     {len(normal_groups)}")
print(f"  Normal (clean):     {(df_seq['disturbance_type']=='Normal').sum()}")
print(f"  With disturbance:   {(df_seq['disturbance_type']!='Normal').sum()}")
print(f"")
print(f"ML sigma (all sequences):")
print(f"  Median: {df_seq['ML_std'].median():.3f} mm")
print(f"  Mean:   {df_seq['ML_std'].mean():.3f} mm")
print(f"  Stable (< {CONFIG.ml_stability_threshold_mm} mm): {100*(df_seq['ML_std'] < CONFIG.ml_stability_threshold_mm).mean():.1f}%")
print(f"")
print(f"ML sigma (clean only):")
print(f"  Median: {df_clean['ML_std'].median():.3f} mm")
print(f"  Stable: {100*(df_clean['ML_std'] < CONFIG.ml_stability_threshold_mm).mean():.1f}%")
print(f"")
print(f"Key DataFrames available:")
print(f"  df        — full Pandas DataFrame ({len(df):,} rows)")
print(f"  df_fc     — FC Mold active subset ({len(df_fc):,} rows)")
print(f"  df_seq    — per-sequence statistics ({len(df_seq)} sequences)")
print(f"  df_clean  — clean (Normal) sequences only ({len(df_clean)} sequences)")